# PAPER-EXP-04A — Conventional Image-Level Internal Evaluation

This is the predeclared methodological comparison to finalized **PAPER-EXP-01C**.

**Important:** this split is deliberately **not lesion-safe**.

The frozen image-level split contains:
- TRAIN: **8,003 images**
- VALIDATION: **1,041 images**
- TEST: **971 images**

It is class-stratified but **not grouped by lesion**. Exact image/file/pixel hashes
remain disjoint, while lesion identities intentionally cross partitions.

Frozen overlap:
- TRAIN ↔ VALIDATION: **358 shared lesions**
- TRAIN ↔ TEST: **360 shared lesions**
- VALIDATION ↔ TEST: **54 shared lesions**
- TEST images whose lesion appears in TRAIN: **372 / 971 (38.31%)**
- TEST images whose lesion appears in TRAIN or VALIDATION: **402 / 971 (41.40%)**

The model recipe is intentionally matched to the finalized clean model:
**3× EfficientNet-B0 + 3× EfficientNet-B3, 300 px, fixed TTA7,
equal seed averaging, fixed 50/50 family fusion.**

No test-guided tuning is allowed.


In [1]:
from pathlib import Path

SCRIPT = 'from __future__ import annotations\n\nimport contextlib\nimport hashlib\nimport io\nimport json\nimport math\nimport platform\nimport random\nimport shutil\nimport sys\nimport zipfile\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageOps\n\nimport sklearn\nfrom sklearn.metrics import (\n    accuracy_score,\n    average_precision_score,\n    balanced_accuracy_score,\n    confusion_matrix,\n    f1_score,\n    precision_recall_curve,\n    precision_recall_fscore_support,\n    recall_score,\n    roc_auc_score,\n    roc_curve,\n)\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torchvision\nfrom torch.utils.data import DataLoader, Dataset, WeightedRandomSampler\nfrom torchvision import models, transforms\nfrom tqdm.auto import tqdm\n\n\nCODE_VERSION = "PAPEREXP04A_CONVENTIONAL_IMAGE_LEVEL_B0B3_300_FIXED_FUSION_2026-08-24_v1"\nINPUT_ROOT = Path("/kaggle/input")\nWORKING_ROOT = Path("/kaggle/working")\nOUT = WORKING_ROOT / "PAPEREXP04A_Conventional_ImageLevel"\nFIG = OUT / "figures"\nSRC = OUT / "figure_source_data"\nTAB = OUT / "tables"\nAUDIT = OUT / "audit"\nCKPT = OUT / "checkpoints"\nfor p in (OUT, FIG, SRC, TAB, AUDIT, CKPT):\n    p.mkdir(parents=True, exist_ok=True)\n\nCLASS_NAMES = ("MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC")\nNUM_CLASSES = 7\nMEL_INDEX = 0\nSEEDS = (20260820, 20260821, 20260822)\n\nIMAGE_SIZE = 300\nMAX_EPOCHS = 30\nLR = 1e-4\nWEIGHT_DECAY = 1e-4\nLABEL_SMOOTHING = 0.05\nB0_BATCH = 64\nB3_BATCH = 32\nEVAL_BATCH = 64\nNUM_WORKERS = 4\n\nTRAIN_MANIFEST_SHA = "dd4ffe8cf5ba0bbe01a2e0f1e0a276d29b9407f2bf8f69f1d045f3a134831a85"\nVAL_MANIFEST_SHA = "6a8a4860f775c29287e5d5a332ef999b3693f71c46d5f45dd84a8824d2fa0df6"\nTEST_MANIFEST_SHA = "4bdb74fbac011be886585aa906a31007f88333a6d7372f1c266e85d16b8505ce"\nFULL_MANIFEST_SHA = "45c425c441e321327b69292475ef1b7d83903860a84b92c979a4ebaf7c844f62"\nSPLIT_FREEZE_ZIP_SHA = "6b8267e59d1b1bf64dde1119c12611f228bda0d0df8dbe3ccbf9065cd5e15d7c"\n\nB0_WEIGHT_SHA = "7f5810bc96def8f7552d5b7e68d53c4786f81167d28291b21c0d90e1fca14934"\nB3_WEIGHT_SHA = "b3899882250c22946d0229d266049fcd133c169233530b36b9ffa7983988362f"\nB0_WEIGHT_FILE = "efficientnet_b0_rwightman-7f5810bc.pth"\nB3_WEIGHT_FILE = "efficientnet_b3_rwightman-b3899882.pth"\nB0_WEIGHT_URL = "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth"\nB3_WEIGHT_URL = "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth"\n\nCLEAN_01C = {\'accuracy\': 0.8682, \'macro_f1\': 0.7333, \'balanced_accuracy\': 0.7076, \'melanoma_recall\': 0.7524, \'macro_auc\': 0.9756}\n\nEXPECTED_OVERLAP = {\n    "train_validation_lesions": 358,\n    "train_test_lesions": 360,\n    "validation_test_lesions": 54,\n    "validation_images_seen_in_train": 373,\n    "test_images_seen_in_train": 372,\n    "test_images_seen_in_train_or_validation": 402,\n}\n\nBOOTSTRAP_ITERS = 2000\nBOOTSTRAP_SEED = 20260824\nIMG_EXTS = {".jpg", ".jpeg", ".png"}\n\n\ndef sha256_bytes(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\n\ndef sha256_file(path: Path, block_size: int = 1 << 20) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as f:\n        for chunk in iter(lambda: f.read(block_size), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef set_seed(seed: int):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = True\n\n\n@contextlib.contextmanager\ndef amp_context(device):\n    if device.type == "cuda":\n        with torch.autocast(device_type="cuda", dtype=torch.float16):\n            yield\n    else:\n        yield\n\n\ndef make_scaler(device):\n    if device.type != "cuda":\n        return None\n    try:\n        return torch.amp.GradScaler("cuda", enabled=True)\n    except Exception:\n        return torch.cuda.amp.GradScaler(enabled=True)\n\n\ndef locate_manifest(filename: str, expected_sha: str):\n    # Direct file\n    for p in INPUT_ROOT.rglob(filename):\n        if p.is_file():\n            raw = p.read_bytes()\n            if sha256_bytes(raw) == expected_sha:\n                return pd.read_csv(io.BytesIO(raw)), str(p)\n\n    # ZIP member\n    for zp in INPUT_ROOT.rglob("*.zip"):\n        if not zp.is_file():\n            continue\n        try:\n            with zipfile.ZipFile(zp) as z:\n                for member in z.namelist():\n                    if Path(member).name != filename:\n                        continue\n                    raw = z.read(member)\n                    if sha256_bytes(raw) == expected_sha:\n                        return pd.read_csv(io.BytesIO(raw)), f"{zp}::{member}"\n        except zipfile.BadZipFile:\n            continue\n\n    raise FileNotFoundError(\n        f"Could not find exact frozen manifest {filename} with SHA256 {expected_sha}. "\n        "Attach PAPEREXP04A_ImageLevel_Split_Freeze.zip."\n    )\n\n\ndef load_frozen_split():\n    train, s_train = locate_manifest(\n        "paperexp04a_imagelevel_train_manifest.csv", TRAIN_MANIFEST_SHA\n    )\n    val, s_val = locate_manifest(\n        "paperexp04a_imagelevel_validation_manifest.csv", VAL_MANIFEST_SHA\n    )\n    test, s_test = locate_manifest(\n        "paperexp04a_imagelevel_test_manifest.csv", TEST_MANIFEST_SHA\n    )\n\n    if len(train) != 8003 or len(val) != 1041 or len(test) != 971:\n        raise RuntimeError("Frozen image-level split sizes changed.")\n    if len(set(train.image_id) & set(val.image_id)):\n        raise RuntimeError("Image ID overlap train/validation.")\n    if len(set(train.image_id) & set(test.image_id)):\n        raise RuntimeError("Image ID overlap train/test.")\n    if len(set(val.image_id) & set(test.image_id)):\n        raise RuntimeError("Image ID overlap validation/test.")\n\n    def overlap(a, b, col):\n        return len(set(a[col].astype(str)) & set(b[col].astype(str)))\n\n    pair = {\n        "train_validation": {\n            "image_id": overlap(train, val, "image_id"),\n            "lesion_id": overlap(train, val, "lesion_id"),\n            "file_sha256": overlap(train, val, "file_sha256"),\n            "pixel_sha256": overlap(train, val, "pixel_sha256"),\n        },\n        "train_test": {\n            "image_id": overlap(train, test, "image_id"),\n            "lesion_id": overlap(train, test, "lesion_id"),\n            "file_sha256": overlap(train, test, "file_sha256"),\n            "pixel_sha256": overlap(train, test, "pixel_sha256"),\n        },\n        "validation_test": {\n            "image_id": overlap(val, test, "image_id"),\n            "lesion_id": overlap(val, test, "lesion_id"),\n            "file_sha256": overlap(val, test, "file_sha256"),\n            "pixel_sha256": overlap(val, test, "pixel_sha256"),\n        },\n    }\n\n    if pair["train_validation"]["lesion_id"] != EXPECTED_OVERLAP["train_validation_lesions"]:\n        raise RuntimeError("Frozen train/validation lesion overlap changed.")\n    if pair["train_test"]["lesion_id"] != EXPECTED_OVERLAP["train_test_lesions"]:\n        raise RuntimeError("Frozen train/test lesion overlap changed.")\n    if pair["validation_test"]["lesion_id"] != EXPECTED_OVERLAP["validation_test_lesions"]:\n        raise RuntimeError("Frozen validation/test lesion overlap changed.")\n\n    for p in pair.values():\n        if p["image_id"] or p["file_sha256"] or p["pixel_sha256"]:\n            raise RuntimeError("Unexpected exact image/hash overlap in frozen split.")\n\n    train_lesions = set(train["lesion_id"].astype(str))\n    val_lesions = set(val["lesion_id"].astype(str))\n    val_seen = val["lesion_id"].astype(str).isin(train_lesions)\n    test_seen_train = test["lesion_id"].astype(str).isin(train_lesions)\n    test_seen_dev = test["lesion_id"].astype(str).isin(train_lesions | val_lesions)\n\n    if int(val_seen.sum()) != EXPECTED_OVERLAP["validation_images_seen_in_train"]:\n        raise RuntimeError("Frozen validation seen-lesion image count changed.")\n    if int(test_seen_train.sum()) != EXPECTED_OVERLAP["test_images_seen_in_train"]:\n        raise RuntimeError("Frozen test train-seen lesion image count changed.")\n    if int(test_seen_dev.sum()) != EXPECTED_OVERLAP["test_images_seen_in_train_or_validation"]:\n        raise RuntimeError("Frozen test development-seen lesion image count changed.")\n\n    audit = {\n        "train_source": s_train,\n        "validation_source": s_val,\n        "test_source": s_test,\n        "pair_overlap": pair,\n        "validation_images_seen_in_train": int(val_seen.sum()),\n        "test_images_seen_in_train": int(test_seen_train.sum()),\n        "test_images_seen_in_train_or_validation": int(test_seen_dev.sum()),\n        "test_fraction_seen_in_train": float(test_seen_train.mean()),\n        "test_fraction_seen_in_train_or_validation": float(test_seen_dev.mean()),\n    }\n    return train, val, test, audit\n\n\ndef find_or_download_weight(filename: str, expected_sha: str, url: str):\n    for p in INPUT_ROOT.rglob(filename):\n        if p.is_file() and sha256_file(p) == expected_sha:\n            print("[WEIGHT] attached", p)\n            return p, "attached"\n\n    cache = WORKING_ROOT / "paperexp04a_pretrained_weights"\n    cache.mkdir(parents=True, exist_ok=True)\n    p = cache / filename\n\n    if p.exists() and sha256_file(p) == expected_sha:\n        return p, "working_cache"\n    if p.exists():\n        p.unlink()\n\n    print("[WEIGHT] downloading official Torchvision weight:", filename)\n    try:\n        torch.hub.download_url_to_file(url, str(p), progress=True)\n    except Exception as exc:\n        raise FileNotFoundError(\n            f"{filename} is not attached and official download failed. "\n            "Turn Kaggle Internet ON or attach the exact pretrained weight file."\n        ) from exc\n\n    digest = sha256_file(p)\n    if digest != expected_sha:\n        p.unlink(missing_ok=True)\n        raise RuntimeError(\n            f"Pretrained weight hash mismatch for {filename}: "\n            f"expected {expected_sha}, got {digest}"\n        )\n    return p, "official_download"\n\n\ndef index_images(frames):\n    required = set()\n    for df in frames:\n        required.update(df["image_id"].astype(str))\n\n    found = {}\n    for p in INPUT_ROOT.rglob("*"):\n        if not p.is_file() or p.suffix.lower() not in IMG_EXTS:\n            continue\n        stem = p.stem\n        if stem in required and stem not in found:\n            found[stem] = p\n\n    missing = sorted(required - set(found))\n    if missing:\n        raise FileNotFoundError(\n            f"Missing {len(missing)} HAM10000 images. Examples: {missing[:15]}"\n        )\n    return found\n\n\ndef make_transform(train: bool):\n    if train:\n        return transforms.Compose([\n            transforms.RandomResizedCrop(\n                IMAGE_SIZE,\n                scale=(0.80, 1.00),\n                ratio=(0.90, 1.10),\n                interpolation=transforms.InterpolationMode.BILINEAR,\n            ),\n            transforms.RandomHorizontalFlip(0.5),\n            transforms.RandomVerticalFlip(0.5),\n            transforms.RandomRotation(\n                180,\n                interpolation=transforms.InterpolationMode.BILINEAR,\n                fill=0,\n            ),\n            transforms.ColorJitter(\n                brightness=0.15,\n                contrast=0.15,\n                saturation=0.10,\n                hue=0.02,\n            ),\n            transforms.ToTensor(),\n            transforms.Normalize(\n                [0.485, 0.456, 0.406],\n                [0.229, 0.224, 0.225],\n            ),\n        ])\n    return transforms.Compose([\n        transforms.Resize(\n            (IMAGE_SIZE, IMAGE_SIZE),\n            interpolation=transforms.InterpolationMode.BILINEAR,\n        ),\n        transforms.ToTensor(),\n        transforms.Normalize(\n            [0.485, 0.456, 0.406],\n            [0.229, 0.224, 0.225],\n        ),\n    ])\n\n\nclass SkinDataset(Dataset):\n    def __init__(self, frame, image_index, train=False):\n        self.frame = frame.reset_index(drop=True)\n        self.image_index = image_index\n        self.transform = make_transform(train)\n\n    def __len__(self):\n        return len(self.frame)\n\n    def __getitem__(self, idx):\n        row = self.frame.iloc[idx]\n        with Image.open(self.image_index[str(row["image_id"])]) as im:\n            im = ImageOps.exif_transpose(im).convert("RGB")\n            x = self.transform(im)\n        return x, int(row["label"]), str(row["image_id"]), str(row["lesion_id"])\n\n\ndef make_sampler(train_df, seed):\n    counts = train_df["label"].value_counts().to_dict()\n    class_weight = {int(k): 1.0 / math.sqrt(float(v)) for k, v in counts.items()}\n    sample_weight = train_df["label"].map(class_weight).astype(float).to_numpy()\n    generator = torch.Generator()\n    generator.manual_seed(seed)\n    return WeightedRandomSampler(\n        torch.as_tensor(sample_weight, dtype=torch.double),\n        num_samples=len(sample_weight),\n        replacement=True,\n        generator=generator,\n    )\n\n\ndef build_imagenet_model(family, weight_path):\n    if family == "B0":\n        model = models.efficientnet_b0(weights=None)\n    elif family == "B3":\n        model = models.efficientnet_b3(weights=None)\n    else:\n        raise ValueError(family)\n\n    state = torch.load(weight_path, map_location="cpu", weights_only=False)\n    if isinstance(state, dict) and "state_dict" in state:\n        state = state["state_dict"]\n    if state and all(str(k).startswith("module.") for k in state):\n        state = {str(k)[7:]: v for k, v in state.items()}\n\n    missing, unexpected = model.load_state_dict(state, strict=False)\n    if missing or unexpected:\n        raise RuntimeError(\n            f"{family} ImageNet weight mismatch: missing={missing}, unexpected={unexpected}"\n        )\n    model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)\n    return model\n\n\ndef build_model_from_checkpoint(path, family):\n    obj = torch.load(path, map_location="cpu", weights_only=False)\n    if family == "B0":\n        model = models.efficientnet_b0(weights=None)\n    else:\n        model = models.efficientnet_b3(weights=None)\n    model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)\n    model.load_state_dict(obj["model_state_dict"], strict=True)\n    return model, obj\n\n\ndef core_metrics(y, probs):\n    y = np.asarray(y, dtype=int)\n    probs = np.asarray(probs, dtype=float)\n    pred = probs.argmax(axis=1)\n    out = {\n        "accuracy": float(accuracy_score(y, pred)),\n        "macro_f1": float(\n            f1_score(y, pred, labels=list(range(NUM_CLASSES)), average="macro", zero_division=0)\n        ),\n        "balanced_accuracy": float(balanced_accuracy_score(y, pred)),\n        "melanoma_recall": float(\n            recall_score(y, pred, labels=[MEL_INDEX], average="macro", zero_division=0)\n        ),\n    }\n    try:\n        out["macro_auc"] = float(\n            roc_auc_score(\n                y,\n                probs,\n                labels=list(range(NUM_CLASSES)),\n                multi_class="ovr",\n                average="macro",\n            )\n        )\n    except Exception:\n        out["macro_auc"] = float("nan")\n    return out\n\n\ndef detailed_metrics(y, probs):\n    y = np.asarray(y, dtype=int)\n    probs = np.asarray(probs, dtype=float)\n    pred = probs.argmax(axis=1)\n    cm = confusion_matrix(y, pred, labels=list(range(NUM_CLASSES)))\n    precision, recall, f1, support = precision_recall_fscore_support(\n        y, pred, labels=list(range(NUM_CLASSES)), zero_division=0\n    )\n    rows = []\n    for i, c in enumerate(CLASS_NAMES):\n        tp = cm[i, i]\n        fn = cm[i, :].sum() - tp\n        fp = cm[:, i].sum() - tp\n        tn = cm.sum() - tp - fn - fp\n        binary = (y == i).astype(int)\n        try:\n            auc = float(roc_auc_score(binary, probs[:, i]))\n            ap = float(average_precision_score(binary, probs[:, i]))\n        except Exception:\n            auc, ap = float("nan"), float("nan")\n        rows.append({\n            "class_name": c,\n            "precision": float(precision[i]),\n            "recall": float(recall[i]),\n            "specificity": float(tn / (tn + fp)) if (tn + fp) else float("nan"),\n            "f1": float(f1[i]),\n            "support": int(support[i]),\n            "roc_auc": auc,\n            "average_precision": ap,\n        })\n    return pd.DataFrame(rows), cm\n\n\n@torch.no_grad()\ndef predict(model, loader, device, tta7=False, desc=None):\n    model.eval()\n    ys, probs, ids, lesions = [], [], [], []\n    for x, y, image_ids, lesion_ids in tqdm(loader, desc=desc, leave=False):\n        x = x.to(device, non_blocking=True)\n        views = (\n            (x,)\n            if not tta7\n            else (\n                x,\n                torch.flip(x, dims=(3,)),\n                torch.flip(x, dims=(2,)),\n                torch.flip(x, dims=(2, 3)),\n                torch.rot90(x, 1, dims=(2, 3)),\n                torch.rot90(x, 2, dims=(2, 3)),\n                torch.rot90(x, 3, dims=(2, 3)),\n            )\n        )\n        total = None\n        for view in views:\n            with amp_context(device):\n                logits = model(view)\n            p = F.softmax(logits.float(), dim=1)\n            total = p if total is None else total + p\n        total = total / len(views)\n        ys.append(y.numpy())\n        probs.append(total.cpu().numpy())\n        ids.extend(list(image_ids))\n        lesions.extend(list(lesion_ids))\n    return np.concatenate(ys), np.concatenate(probs), ids, lesions\n\n\ndef train_seed(family, seed, train_df, val_df, image_index, weight_path, device):\n    set_seed(seed)\n    batch = B0_BATCH if family == "B0" else B3_BATCH\n\n    train_loader = DataLoader(\n        SkinDataset(train_df, image_index, train=True),\n        batch_size=batch,\n        sampler=make_sampler(train_df, seed),\n        num_workers=NUM_WORKERS,\n        pin_memory=True,\n        persistent_workers=NUM_WORKERS > 0,\n        drop_last=False,\n    )\n    val_loader = DataLoader(\n        SkinDataset(val_df, image_index, train=False),\n        batch_size=EVAL_BATCH,\n        shuffle=False,\n        num_workers=NUM_WORKERS,\n        pin_memory=True,\n        persistent_workers=NUM_WORKERS > 0,\n        drop_last=False,\n    )\n\n    model = build_imagenet_model(family, weight_path).to(device)\n    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(\n        optimizer, T_max=MAX_EPOCHS, eta_min=LR * 0.05\n    )\n    scaler = make_scaler(device)\n\n    family_dir = CKPT / family / f"seed_{seed}"\n    family_dir.mkdir(parents=True, exist_ok=True)\n    best = None\n    history = []\n\n    for epoch in range(1, MAX_EPOCHS + 1):\n        model.train()\n        loss_sum, seen = 0.0, 0\n\n        for x, y, _, _ in tqdm(\n            train_loader,\n            desc=f"{family}-300 seed {seed} epoch {epoch}/{MAX_EPOCHS}",\n            leave=False,\n        ):\n            x = x.to(device, non_blocking=True)\n            y = y.to(device, non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n\n            with amp_context(device):\n                logits = model(x)\n                loss = criterion(logits, y)\n\n            if scaler is not None:\n                scaler.scale(loss).backward()\n                scaler.step(optimizer)\n                scaler.update()\n            else:\n                loss.backward()\n                optimizer.step()\n\n            loss_sum += float(loss.detach().cpu()) * x.size(0)\n            seen += x.size(0)\n\n        yv, pv, _, _ = predict(\n            model, val_loader, device, tta7=False,\n            desc=f"{family} validation original seed {seed} epoch {epoch}"\n        )\n        vm = core_metrics(yv, pv)\n        row = {\n            "family": family,\n            "seed": seed,\n            "epoch": epoch,\n            "train_loss": loss_sum / max(1, seen),\n            "lr_before_scheduler_step": float(optimizer.param_groups[0]["lr"]),\n            **vm,\n        }\n        history.append(row)\n        print(json.dumps(row))\n\n        key = (vm["macro_f1"], vm["accuracy"], -epoch)\n        if best is None or key > best["key"]:\n            ckpt_path = family_dir / f"best_{family.lower()}_300.pth"\n            torch.save({\n                "experiment": "PAPER-EXP-04A",\n                "code_version": CODE_VERSION,\n                "family": family,\n                "seed": seed,\n                "epoch": epoch,\n                "resolution": IMAGE_SIZE,\n                "model_state_dict": {\n                    k: v.detach().cpu()\n                    for k, v in model.state_dict().items()\n                },\n                "validation_metrics_original": vm,\n                "train_manifest_sha256": TRAIN_MANIFEST_SHA,\n                "validation_manifest_sha256": VAL_MANIFEST_SHA,\n                "test_manifest_sha256": TEST_MANIFEST_SHA,\n                "test_access_during_training": False,\n                "class_order": list(CLASS_NAMES),\n                "checkpoint_selection": (\n                    "validation original-view Macro-F1; accuracy tie-breaker; "\n                    "earlier epoch final tie-breaker"\n                ),\n            }, ckpt_path)\n            best = {\n                "key": key,\n                "path": ckpt_path,\n                "epoch": epoch,\n                "metrics": vm,\n            }\n\n        scheduler.step()\n\n    pd.DataFrame(history).to_csv(family_dir / "training_history.csv", index=False)\n\n    model2, obj = build_model_from_checkpoint(best["path"], family)\n    model2 = model2.to(device).eval()\n    y7, p7, ids7, lesions7 = predict(\n        model2, val_loader, device, tta7=True,\n        desc=f"{family} seed {seed} fixed TTA7 validation"\n    )\n\n    result = {\n        "family": family,\n        "seed": seed,\n        "best_epoch": best["epoch"],\n        "checkpoint_path": best["path"],\n        "checkpoint_sha256": sha256_file(best["path"]),\n        "validation_tta7_metrics": core_metrics(y7, p7),\n        "y_val": y7,\n        "p_val": p7,\n        "ids_val": ids7,\n        "lesions_val": lesions7,\n    }\n\n    del model, model2\n    torch.cuda.empty_cache()\n    return result\n\n\ndef save_predictions(path, frame, probs, extra=None):\n    y = frame["label"].to_numpy(int)\n    pred = probs.argmax(axis=1)\n    out = pd.DataFrame({\n        "image_id": frame["image_id"].astype(str),\n        "lesion_id": frame["lesion_id"].astype(str),\n        "true_label": y,\n        "true_class": frame["class_name"].astype(str),\n        "predicted_label": pred,\n        "predicted_class": [CLASS_NAMES[i] for i in pred],\n        "correct": (pred == y).astype(int),\n    })\n    if extra:\n        for k, v in extra.items():\n            out[k] = v\n    for i, c in enumerate(CLASS_NAMES):\n        out[f"prob_{c}"] = probs[:, i]\n    out.to_csv(path, index=False)\n    return out\n\n\ndef lesion_bootstrap(y, probs, lesions):\n    y = np.asarray(y, dtype=int)\n    probs = np.asarray(probs, dtype=float)\n    lesions = np.asarray(lesions, dtype=str)\n\n    unique = np.unique(lesions)\n    by_lesion = {g: np.flatnonzero(lesions == g) for g in unique}\n    rng = np.random.default_rng(BOOTSTRAP_SEED)\n    rows = []\n\n    for draw in tqdm(range(BOOTSTRAP_ITERS), desc="TEST lesion bootstrap"):\n        sampled = rng.choice(unique, size=len(unique), replace=True)\n        idx = np.concatenate([by_lesion[g] for g in sampled])\n        yy = y[idx]\n        if len(np.unique(yy)) < NUM_CLASSES:\n            continue\n        try:\n            mm = core_metrics(yy, probs[idx])\n        except Exception:\n            continue\n        if not np.isfinite(mm["macro_auc"]):\n            continue\n        rows.append({"draw": draw, "n_images": int(len(idx)), **mm})\n\n    boot = pd.DataFrame(rows)\n    summary = []\n    point = core_metrics(y, probs)\n    for metric in ("accuracy", "macro_f1", "balanced_accuracy", "melanoma_recall", "macro_auc"):\n        vals = boot[metric].to_numpy(float)\n        summary.append({\n            "metric": metric,\n            "point": point[metric],\n            "ci95_lower": float(np.percentile(vals, 2.5)),\n            "ci95_upper": float(np.percentile(vals, 97.5)),\n            "valid_draws": int(len(vals)),\n            "requested_draws": BOOTSTRAP_ITERS,\n        })\n    return boot, pd.DataFrame(summary)\n\n\ndef save_figure(fig, stem):\n    fig.tight_layout()\n    fig.savefig(FIG / f"{stem}.png", dpi=300, bbox_inches="tight")\n    fig.savefig(FIG / f"{stem}.pdf", bbox_inches="tight")\n    fig.savefig(FIG / f"{stem}.svg", bbox_inches="tight")\n    plt.close(fig)\n\n\ndef make_publication_package(test_df, probs, ci_table):\n    y = test_df["label"].to_numpy(int)\n    pc, cm = detailed_metrics(y, probs)\n    pc.to_csv(TAB / "table_test_per_class_metrics.csv", index=False)\n\n    cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)\n    cm_df.to_csv(SRC / "test_confusion_matrix_counts.csv")\n    cm_norm = cm / cm.sum(axis=1, keepdims=True)\n    pd.DataFrame(cm_norm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(\n        SRC / "test_confusion_matrix_row_normalized.csv"\n    )\n\n    for data, suffix, integer in (\n        (cm, "counts", True),\n        (cm_norm, "row_normalized", False),\n    ):\n        fig, ax = plt.subplots(figsize=(8, 7))\n        im = ax.imshow(data, aspect="auto")\n        ax.set_xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=45, ha="right")\n        ax.set_yticks(range(NUM_CLASSES), CLASS_NAMES)\n        ax.set_xlabel("Predicted class")\n        ax.set_ylabel("True class")\n        ax.set_title(f"Conventional image-level TEST confusion matrix ({suffix})")\n        for i in range(NUM_CLASSES):\n            for j in range(NUM_CLASSES):\n                txt = f"{int(data[i, j])}" if integer else f"{data[i, j]:.2f}"\n                ax.text(j, i, txt, ha="center", va="center", fontsize=8)\n        fig.colorbar(im, ax=ax)\n        save_figure(fig, "Fig1_confusion_matrix_normalized" if not integer else "FigS1_confusion_matrix_counts")\n\n    # ROC/PR\n    roc_rows, pr_rows = [], []\n    fig_roc, ax_roc = plt.subplots(figsize=(8, 7))\n    fig_pr, ax_pr = plt.subplots(figsize=(8, 7))\n    for i, c in enumerate(CLASS_NAMES):\n        binary = (y == i).astype(int)\n        fpr, tpr, _ = roc_curve(binary, probs[:, i])\n        precision, recall, _ = precision_recall_curve(binary, probs[:, i])\n        auc = roc_auc_score(binary, probs[:, i])\n        ap = average_precision_score(binary, probs[:, i])\n\n        ax_roc.plot(fpr, tpr, label=f"{c} (AUC {auc:.3f})")\n        ax_pr.plot(recall, precision, label=f"{c} (AP {ap:.3f})")\n\n        for a, b in zip(fpr, tpr):\n            roc_rows.append({"class_name": c, "fpr": float(a), "tpr": float(b), "roc_auc": float(auc)})\n        for a, b in zip(recall, precision):\n            pr_rows.append({"class_name": c, "recall": float(a), "precision": float(b), "average_precision": float(ap)})\n\n    ax_roc.plot([0, 1], [0, 1], linestyle="--")\n    ax_roc.set_xlabel("False positive rate")\n    ax_roc.set_ylabel("True positive rate")\n    ax_roc.set_title("Conventional image-level TEST ROC curves")\n    ax_roc.legend(fontsize=8)\n    save_figure(fig_roc, "Fig2_multiclass_ROC")\n\n    ax_pr.set_xlabel("Recall")\n    ax_pr.set_ylabel("Precision")\n    ax_pr.set_title("Conventional image-level TEST precision-recall curves")\n    ax_pr.legend(fontsize=8)\n    save_figure(fig_pr, "Fig3_multiclass_precision_recall")\n\n    pd.DataFrame(roc_rows).to_csv(SRC / "roc_curve_data.csv", index=False)\n    pd.DataFrame(pr_rows).to_csv(SRC / "precision_recall_curve_data.csv", index=False)\n\n    # Per-class figure\n    ordered = pc.set_index("class_name").reindex(CLASS_NAMES)\n    x = np.arange(NUM_CLASSES)\n    w = 0.20\n    fig, ax = plt.subplots(figsize=(12, 6))\n    ax.bar(x - 1.5*w, ordered["recall"], w, label="Recall")\n    ax.bar(x - 0.5*w, ordered["f1"], w, label="F1")\n    ax.bar(x + 0.5*w, ordered["roc_auc"], w, label="ROC-AUC")\n    ax.bar(x + 1.5*w, ordered["average_precision"], w, label="AP")\n    ax.set_xticks(x, CLASS_NAMES)\n    ax.set_ylim(0, 1)\n    ax.set_ylabel("Score")\n    ax.set_title("Conventional image-level TEST per-class performance")\n    ax.legend()\n    save_figure(fig, "Fig4_per_class_performance")\n\n    # Primary metrics + CI\n    order = ["accuracy", "macro_f1", "balanced_accuracy", "melanoma_recall", "macro_auc"]\n    ci = ci_table.set_index("metric").loc[order].reset_index()\n    xv = np.arange(len(ci))\n    point = ci["point"].to_numpy(float)\n    lo = point - ci["ci95_lower"].to_numpy(float)\n    hi = ci["ci95_upper"].to_numpy(float) - point\n    fig, ax = plt.subplots(figsize=(10, 5.5))\n    ax.errorbar(xv, point, yerr=np.vstack([lo, hi]), fmt="o", capsize=5)\n    ax.set_xticks(xv, [m.replace("_", "\\n") for m in ci["metric"]])\n    ax.set_ylim(0, 1)\n    ax.set_ylabel("Score")\n    ax.set_title("Conventional image-level TEST metrics with lesion-bootstrap 95% CI")\n    save_figure(fig, "Fig5_primary_metrics_95CI")\n\n    # Clean vs conventional descriptive comparison.\n    point_metrics = core_metrics(y, probs)\n    comp = pd.DataFrame({\n        "metric": order,\n        "PAPEREXP01C_lesion_safe": [CLEAN_01C[m] for m in order],\n        "PAPEREXP04A_conventional_image_level": [point_metrics[m] for m in order],\n    })\n    comp["descriptive_delta_04A_minus_01C"] = (\n        comp["PAPEREXP04A_conventional_image_level"] - comp["PAPEREXP01C_lesion_safe"]\n    )\n    comp["comparison_note"] = (\n        "Different frozen split protocols; descriptive comparison, not paired."\n    )\n    comp.to_csv(TAB / "table_clean01C_vs_conventional04A_descriptive.csv", index=False)\n\n    fig, ax = plt.subplots(figsize=(11, 6))\n    w = 0.36\n    ax.bar(xv - w/2, comp["PAPEREXP01C_lesion_safe"], w, label="PAPER-EXP-01C lesion-safe")\n    ax.bar(xv + w/2, comp["PAPEREXP04A_conventional_image_level"], w, label="PAPER-EXP-04A image-level")\n    ax.set_xticks(xv, [m.replace("_", "\\n") for m in comp["metric"]])\n    ax.set_ylim(0, 1)\n    ax.set_ylabel("Score")\n    ax.set_title("Lesion-safe vs conventional image-level internal evaluation")\n    ax.legend()\n    save_figure(fig, "Fig6_lesion_safe_vs_image_level")\n\n    # Class distribution\n    dist = (\n        test_df.groupby(["label", "class_name"], as_index=False)\n        .agg(images=("image_id", "size"), unique_lesions=("lesion_id", "nunique"))\n        .sort_values("label")\n    )\n    dist.to_csv(TAB / "table_test_class_distribution.csv", index=False)\n    fig, ax = plt.subplots(figsize=(9, 5.5))\n    ax.bar(np.arange(len(dist)), dist["images"])\n    ax.set_xticks(np.arange(len(dist)), dist["class_name"])\n    ax.set_ylabel("Images")\n    ax.set_title("Conventional image-level TEST class distribution")\n    save_figure(fig, "FigS2_test_class_distribution")\n\n\ndef leakage_subgroup_analysis(train_df, val_df, test_df, probs):\n    y = test_df["label"].to_numpy(int)\n    train_lesions = set(train_df["lesion_id"].astype(str))\n    val_lesions = set(val_df["lesion_id"].astype(str))\n    lesion_ids = test_df["lesion_id"].astype(str)\n\n    masks = {\n        "test_all": np.ones(len(test_df), dtype=bool),\n        "lesion_seen_in_train": lesion_ids.isin(train_lesions).to_numpy(),\n        "lesion_not_seen_in_train": (~lesion_ids.isin(train_lesions)).to_numpy(),\n        "lesion_seen_in_train_or_validation": lesion_ids.isin(train_lesions | val_lesions).to_numpy(),\n        "lesion_unseen_in_train_and_validation": (~lesion_ids.isin(train_lesions | val_lesions)).to_numpy(),\n    }\n\n    rows = []\n    for name, mask in masks.items():\n        yy = y[mask]\n        pp = probs[mask]\n        mm = core_metrics(yy, pp)\n        rows.append({\n            "subset": name,\n            "images": int(mask.sum()),\n            "unique_lesions": int(test_df.loc[mask, "lesion_id"].nunique()),\n            **mm,\n        })\n\n    out = pd.DataFrame(rows)\n    out.to_csv(TAB / "table_lesion_overlap_subgroup_performance.csv", index=False)\n\n    # Accuracy-focused visualization of the direct train-lesion overlap contrast.\n    plot = out[out["subset"].isin(["lesion_seen_in_train", "lesion_not_seen_in_train"])].copy()\n    fig, ax = plt.subplots(figsize=(8, 5.5))\n    ax.bar(plot["subset"], plot["accuracy"])\n    ax.set_ylim(0, 1)\n    ax.set_ylabel("Accuracy")\n    ax.set_title("TEST accuracy by whether lesion identity appeared in TRAIN")\n    ax.tick_params(axis="x", rotation=15)\n    save_figure(fig, "Fig7_seen_vs_unseen_lesion_accuracy")\n\n    return out\n\n\ndef build_material_index():\n    rows = []\n    for p in sorted(OUT.rglob("*")):\n        if not p.is_file():\n            continue\n        rel = str(p.relative_to(OUT))\n        if p.suffix.lower() in {".pth", ".pt", ".ckpt"}:\n            role = "checkpoint_recovery"\n        elif rel.startswith("figures/"):\n            role = "publication_figure"\n        elif rel.startswith("figure_source_data/"):\n            role = "figure_source_data"\n        elif rel.startswith("tables/"):\n            role = "paper_table"\n        elif "prediction" in rel or "probabilities" in rel:\n            role = "prediction_source"\n        elif "bootstrap" in rel:\n            role = "uncertainty_source"\n        else:\n            role = "audit_or_supporting_material"\n\n        rows.append({\n            "file": rel,\n            "role": role,\n            "sha256": sha256_file(p),\n            "bytes": int(p.stat().st_size),\n        })\n    pd.DataFrame(rows).to_csv(OUT / "paper_material_index.csv", index=False)\n    (OUT / "paper_material_index.json").write_text(json.dumps(rows, indent=2), encoding="utf-8")\n\n\ndef main():\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable a Kaggle GPU.")\n    device = torch.device("cuda:0")\n\n    print("[GPU]", torch.cuda.get_device_name(0))\n    print("[CODE VERSION]", CODE_VERSION)\n    print("[SCIENTIFIC STATUS] conventional image-level split; intentionally NON-lesion-safe")\n\n    train_df, val_df, test_df, split_audit = load_frozen_split()\n    image_index = index_images([train_df, val_df, test_df])\n\n    b0_weight, b0_weight_source = find_or_download_weight(\n        B0_WEIGHT_FILE, B0_WEIGHT_SHA, B0_WEIGHT_URL\n    )\n    b3_weight, b3_weight_source = find_or_download_weight(\n        B3_WEIGHT_FILE, B3_WEIGHT_SHA, B3_WEIGHT_URL\n    )\n\n    preflight = {\n        "experiment": "PAPER-EXP-04A",\n        "code_version": CODE_VERSION,\n        "scientific_status": "conventional image-level evaluation (non-lesion-safe)",\n        "split_audit": split_audit,\n        "split_freeze_zip_sha256_expected": SPLIT_FREEZE_ZIP_SHA,\n        "train_manifest_sha256": TRAIN_MANIFEST_SHA,\n        "validation_manifest_sha256": VAL_MANIFEST_SHA,\n        "test_manifest_sha256": TEST_MANIFEST_SHA,\n        "b0_pretrained_source": b0_weight_source,\n        "b0_pretrained_sha256": sha256_file(b0_weight),\n        "b3_pretrained_source": b3_weight_source,\n        "b3_pretrained_sha256": sha256_file(b3_weight),\n        "test_read_during_training": False,\n        "external_data_used": False,\n        "ensemble_weight_search": False,\n        "tta_search": False,\n        "threshold_search": False,\n        "calibration": False,\n    }\n    (AUDIT / "preflight_audit.json").write_text(json.dumps(preflight, indent=2), encoding="utf-8")\n\n    # -----------------------------------------------------------------\n    # TRAIN + VALIDATION selection only.\n    # -----------------------------------------------------------------\n    train_results = []\n    for family, weight_path in (("B0", b0_weight), ("B3", b3_weight)):\n        for seed in SEEDS:\n            train_results.append(\n                train_seed(\n                    family, seed, train_df, val_df,\n                    image_index, weight_path, device\n                )\n            )\n\n    # Aggregate fixed TTA7 validation predictions.\n    val_family_probs = {}\n    seed_summary = []\n\n    for family in ("B0", "B3"):\n        fam = [r for r in train_results if r["family"] == family]\n        y_ref = fam[0]["y_val"]\n        ids_ref = fam[0]["ids_val"]\n        lesions_ref = fam[0]["lesions_val"]\n\n        for r in fam[1:]:\n            if (\n                not np.array_equal(y_ref, r["y_val"])\n                or ids_ref != r["ids_val"]\n                or lesions_ref != r["lesions_val"]\n            ):\n                raise RuntimeError(f"{family} validation row alignment changed.")\n\n        val_family_probs[family] = np.mean([r["p_val"] for r in fam], axis=0)\n\n        for r in fam:\n            seed_summary.append({\n                "family": family,\n                "seed": r["seed"],\n                "best_epoch": r["best_epoch"],\n                "checkpoint_sha256": r["checkpoint_sha256"],\n                "validation_tta7_metrics": r["validation_tta7_metrics"],\n            })\n\n    final_val_probs = 0.5 * val_family_probs["B0"] + 0.5 * val_family_probs["B3"]\n    val_metrics = core_metrics(val_df["label"].to_numpy(int), final_val_probs)\n\n    save_predictions(\n        OUT / "validation_fixed_b0b3_fusion_predictions.csv",\n        val_df,\n        final_val_probs,\n        extra={\n            "lesion_seen_in_train": val_df["lesion_id"].astype(str).isin(\n                set(train_df["lesion_id"].astype(str))\n            ).astype(int)\n        },\n    )\n\n    # -----------------------------------------------------------------\n    # SYSTEM IS NOW LOCKED. TEST IS EVALUATED ONCE.\n    # -----------------------------------------------------------------\n    test_loader = DataLoader(\n        SkinDataset(test_df, image_index, train=False),\n        batch_size=EVAL_BATCH,\n        shuffle=False,\n        num_workers=NUM_WORKERS,\n        pin_memory=True,\n        persistent_workers=NUM_WORKERS > 0,\n        drop_last=False,\n    )\n\n    test_family_probs = {}\n    test_seed_rows = []\n\n    for family in ("B0", "B3"):\n        fam = [r for r in train_results if r["family"] == family]\n        probs_list = []\n\n        for r in fam:\n            model, obj = build_model_from_checkpoint(r["checkpoint_path"], family)\n            model = model.to(device).eval()\n            yt, pt, ids, lesions = predict(\n                model,\n                test_loader,\n                device,\n                tta7=True,\n                desc=f"FROZEN TEST {family} seed {r[\'seed\']} TTA7"\n            )\n\n            if ids != test_df["image_id"].astype(str).tolist():\n                raise RuntimeError("Frozen TEST row order changed.")\n            if lesions != test_df["lesion_id"].astype(str).tolist():\n                raise RuntimeError("Frozen TEST lesion order changed.")\n            if not np.array_equal(yt, test_df["label"].to_numpy(int)):\n                raise RuntimeError("Frozen TEST labels changed.")\n\n            probs_list.append(pt)\n            seed_metric = core_metrics(yt, pt)\n            test_seed_rows.append({\n                "family": family,\n                "seed": r["seed"],\n                "checkpoint_sha256": r["checkpoint_sha256"],\n                **seed_metric,\n            })\n\n            save_predictions(\n                OUT / f"test_{family.lower()}_seed_{r[\'seed\']}_tta7_predictions.csv",\n                test_df,\n                pt,\n            )\n\n            del model\n            torch.cuda.empty_cache()\n\n        test_family_probs[family] = np.mean(probs_list, axis=0)\n        save_predictions(\n            OUT / f"test_{family.lower()}_family_equal3_tta7_predictions.csv",\n            test_df,\n            test_family_probs[family],\n        )\n\n    final_test_probs = 0.5 * test_family_probs["B0"] + 0.5 * test_family_probs["B3"]\n\n    train_lesions = set(train_df["lesion_id"].astype(str))\n    val_lesions = set(val_df["lesion_id"].astype(str))\n    test_extra = {\n        "lesion_seen_in_train": test_df["lesion_id"].astype(str).isin(train_lesions).astype(int),\n        "lesion_seen_in_validation": test_df["lesion_id"].astype(str).isin(val_lesions).astype(int),\n        "lesion_seen_in_train_or_validation": test_df["lesion_id"].astype(str).isin(\n            train_lesions | val_lesions\n        ).astype(int),\n    }\n    final_pred = save_predictions(\n        OUT / "paper_exp04a_conventional_imagelevel_test_predictions.csv",\n        test_df,\n        final_test_probs,\n        extra=test_extra,\n    )\n\n    test_metrics = core_metrics(test_df["label"].to_numpy(int), final_test_probs)\n    per_class, cm = detailed_metrics(test_df["label"].to_numpy(int), final_test_probs)\n    per_class.to_csv(TAB / "table_test_per_class_metrics.csv", index=False)\n    pd.DataFrame(test_seed_rows).to_csv(TAB / "table_test_seed_metrics.csv", index=False)\n\n    boot, ci_table = lesion_bootstrap(\n        test_df["label"].to_numpy(int),\n        final_test_probs,\n        test_df["lesion_id"].astype(str).to_numpy(),\n    )\n    boot.to_csv(OUT / "test_lesion_bootstrap_draws.csv", index=False)\n    ci_table.to_csv(TAB / "table_test_primary_metrics_95CI.csv", index=False)\n\n    subgroup = leakage_subgroup_analysis(\n        train_df, val_df, test_df, final_test_probs\n    )\n\n    make_publication_package(test_df, final_test_probs, ci_table)\n\n    # Save split manifests used in the run into results for self-contained audit.\n    train_df.to_csv(AUDIT / "used_train_manifest.csv", index=False)\n    val_df.to_csv(AUDIT / "used_validation_manifest.csv", index=False)\n    test_df.to_csv(AUDIT / "used_test_manifest.csv", index=False)\n\n    env = {\n        "python": sys.version,\n        "platform": platform.platform(),\n        "torch": torch.__version__,\n        "torchvision": torchvision.__version__,\n        "numpy": np.__version__,\n        "pandas": pd.__version__,\n        "sklearn": sklearn.__version__,\n        "gpu": torch.cuda.get_device_name(0),\n    }\n    (AUDIT / "environment.json").write_text(json.dumps(env, indent=2), encoding="utf-8")\n\n    summary = {\n        "experiment": "PAPER-EXP-04A",\n        "code_version": CODE_VERSION,\n        "scientific_status": "conventional image-level internal evaluation (non-lesion-safe)",\n        "split": {\n            "train_images": 8003,\n            "validation_images": 1041,\n            "test_images": 971,\n            "train_validation_lesion_overlap": split_audit["pair_overlap"]["train_validation"]["lesion_id"],\n            "train_test_lesion_overlap": split_audit["pair_overlap"]["train_test"]["lesion_id"],\n            "validation_test_lesion_overlap": split_audit["pair_overlap"]["validation_test"]["lesion_id"],\n            "validation_images_with_lesion_seen_in_train": split_audit["validation_images_seen_in_train"],\n            "test_images_with_lesion_seen_in_train": split_audit["test_images_seen_in_train"],\n            "test_images_with_lesion_seen_in_train_or_validation": split_audit["test_images_seen_in_train_or_validation"],\n        },\n        "model": {\n            "architecture": "3x EfficientNet-B0 + 3x EfficientNet-B3",\n            "resolution": 300,\n            "inference": "fixed TTA7",\n            "within_family": "equal 1/3",\n            "fusion": "fixed 0.50 B0 + 0.50 B3",\n        },\n        "validation_metrics": val_metrics,\n        "test_metrics": test_metrics,\n        "test_95ci": {\n            row["metric"]: {\n                "lower": float(row["ci95_lower"]),\n                "upper": float(row["ci95_upper"]),\n                "valid_draws": int(row["valid_draws"]),\n            }\n            for _, row in ci_table.iterrows()\n        },\n        "test_subgroup_performance": subgroup.to_dict(orient="records"),\n        "seed_summary": seed_summary,\n        "historical_clean_reference_PAPEREXP01C": CLEAN_01C,\n        "comparison_to_clean_is_descriptive_not_paired": True,\n        "external_data_used": False,\n        "ensemble_weight_search": False,\n        "tta_search": False,\n        "threshold_search": False,\n        "calibration": False,\n        "post_test_model_change_allowed": False,\n        "decision": "REPORT_CONVENTIONAL_IMAGE_LEVEL_RESULT_AS_OBSERVED_NO_POST_TEST_TUNING",\n    }\n    (OUT / "paper_exp04a_final_summary.json").write_text(\n        json.dumps(summary, indent=2), encoding="utf-8"\n    )\n\n    manuscript = f"""PAPER-EXP-04A — CONVENTIONAL IMAGE-LEVEL INTERNAL EVALUATION\n\nProtocol:\nThe same 10,015-image HAM10000 source was repartitioned using a class-stratified\nimage-level split without lesion grouping. Image identities and exact file/pixel\nhashes remain disjoint, but lesion identity may cross TRAIN/VALIDATION/TEST.\n\nIntentional lesion overlap:\nTRAIN-VALIDATION shared lesions: {split_audit[\'pair_overlap\'][\'train_validation\'][\'lesion_id\']}\nTRAIN-TEST shared lesions: {split_audit[\'pair_overlap\'][\'train_test\'][\'lesion_id\']}\nVALIDATION-TEST shared lesions: {split_audit[\'pair_overlap\'][\'validation_test\'][\'lesion_id\']}\nTEST images with lesion seen in TRAIN: {split_audit[\'test_images_seen_in_train\']}/971\nTEST images with lesion seen in TRAIN or VALIDATION: {split_audit[\'test_images_seen_in_train_or_validation\']}/971\n\nModel:\n3 x EfficientNet-B0 + 3 x EfficientNet-B3, 300 px, fixed TTA7,\nequal 1/3 seed averaging within family, fixed 50/50 family fusion.\n\nValidation:\nAccuracy {val_metrics[\'accuracy\']:.6f}\nMacro-F1 {val_metrics[\'macro_f1\']:.6f}\nBalanced accuracy {val_metrics[\'balanced_accuracy\']:.6f}\nMelanoma recall {val_metrics[\'melanoma_recall\']:.6f}\nMacro AUC {val_metrics[\'macro_auc\']:.6f}\n\nFrozen conventional TEST:\nAccuracy {test_metrics[\'accuracy\']:.6f}\nMacro-F1 {test_metrics[\'macro_f1\']:.6f}\nBalanced accuracy {test_metrics[\'balanced_accuracy\']:.6f}\nMelanoma recall {test_metrics[\'melanoma_recall\']:.6f}\nMacro AUC {test_metrics[\'macro_auc\']:.6f}\n\nInterpretation:\nThis is intentionally NOT lesion-safe and is included as a methodological\ncomparison to the finalized lesion-safe PAPER-EXP-01C result. The comparison\nbetween PAPER-EXP-01C and PAPER-EXP-04A is descriptive because the frozen test\npartitions differ. No post-test tuning is permitted.\n"""\n    (OUT / "manuscript_result_block.txt").write_text(manuscript, encoding="utf-8")\n\n    # Save protocol snapshot.\n    protocol_snapshot = {\'experiment\': \'PAPER-EXP-04A\', \'title\': \'Conventional Image-Level Internal Evaluation\', \'code_version\': \'PAPEREXP04A_CONVENTIONAL_IMAGE_LEVEL_B0B3_300_FIXED_FUSION_2026-08-24_v1\', \'scientific_role\': \'Methodological comparison to the finalized lesion-safe PAPER-EXP-01C result. This experiment intentionally uses a non-lesion-safe image-level split.\', \'split_freeze_zip_sha256\': \'6b8267e59d1b1bf64dde1119c12611f228bda0d0df8dbe3ccbf9065cd5e15d7c\', \'split_manifest_hashes\': {\'train_sha256\': \'dd4ffe8cf5ba0bbe01a2e0f1e0a276d29b9407f2bf8f69f1d045f3a134831a85\', \'validation_sha256\': \'6a8a4860f775c29287e5d5a332ef999b3693f71c46d5f45dd84a8824d2fa0df6\', \'test_sha256\': \'4bdb74fbac011be886585aa906a31007f88333a6d7372f1c266e85d16b8505ce\', \'full_sha256\': \'45c425c441e321327b69292475ef1b7d83903860a84b92c979a4ebaf7c844f62\'}, \'split\': {\'experiment\': \'PAPER-EXP-04A\', \'title\': \'Frozen Conventional Image-Level Internal Split\', \'scientific_purpose\': \'Methodological comparison against lesion-safe PAPER-EXP-01C using the same 10015-image HAM10000 source but a conventional class-stratified image-level split.\', \'source_dataset\': {\'images\': 10015, \'unique_lesions\': 7470, \'source_clean_split_zip_sha256\': \'982dc6b1509b28f50f4940db2ac110002971ca0d7ac3aa1c78b07582442fb24f\'}, \'split_method\': {\'grouped_by_lesion\': False, \'stratified_by_class\': True, \'test_size\': 971, \'validation_size\': 1041, \'train_size\': 8003, \'test_seed\': 20260824, \'validation_seed\': 20260825, \'ordering_before_split\': \'image_id ascending\'}, \'manifest_hashes\': {\'train_sha256\': \'dd4ffe8cf5ba0bbe01a2e0f1e0a276d29b9407f2bf8f69f1d045f3a134831a85\', \'validation_sha256\': \'6a8a4860f775c29287e5d5a332ef999b3693f71c46d5f45dd84a8824d2fa0df6\', \'test_sha256\': \'4bdb74fbac011be886585aa906a31007f88333a6d7372f1c266e85d16b8505ce\', \'full_sha256\': \'45c425c441e321327b69292475ef1b7d83903860a84b92c979a4ebaf7c844f62\'}, \'counts\': {\'train_images\': 8003, \'train_unique_lesions\': 6270, \'validation_images\': 1041, \'validation_unique_lesions\': 1002, \'test_images\': 971, \'test_unique_lesions\': 945}, \'partition_overlap_audit\': {\'train_vs_validation\': {\'image_id\': 0, \'lesion_id\': 358, \'split_group\': 358, \'file_sha256\': 0, \'pixel_sha256\': 0}, \'train_vs_test\': {\'image_id\': 0, \'lesion_id\': 360, \'split_group\': 360, \'file_sha256\': 0, \'pixel_sha256\': 0}, \'validation_vs_test\': {\'image_id\': 0, \'lesion_id\': 54, \'split_group\': 54, \'file_sha256\': 0, \'pixel_sha256\': 0}}, \'intentional_lesion_overlap\': {\'validation_images_with_lesion_seen_in_train\': 373, \'validation_fraction_with_lesion_seen_in_train\': 0.35830931796349663, \'test_images_with_lesion_seen_in_train\': 372, \'test_fraction_with_lesion_seen_in_train\': 0.3831101956745623, \'test_images_with_lesion_seen_in_train_or_validation\': 402, \'test_fraction_with_lesion_seen_in_train_or_validation\': 0.4140061791967044, \'test_unique_lesions_seen_in_train\': 360, \'test_unique_lesions_seen_in_train_or_validation\': 389, \'lesions_present_in_all_three_partitions\': 25}, \'interpretation\': \'Image identities and exact file/pixel hashes are disjoint, but lesion identities are intentionally allowed to cross partitions. This is the defining conventional image-level comparison and must not be described as lesion-safe.\'}, \'model\': {\'families\': [\'Torchvision EfficientNet-B0\', \'Torchvision EfficientNet-B3\'], \'resolution\': 300, \'seeds\': [20260820, 20260821, 20260822], \'epochs\': 30, \'B0_batch_size\': 64, \'B3_batch_size\': 32, \'optimizer\': \'AdamW\', \'lr\': 0.0001, \'weight_decay\': 0.0001, \'scheduler\': \'CosineAnnealingLR with eta_min=5e-6\', \'label_smoothing\': 0.05, \'sampler\': \'inverse sqrt image-class frequency WeightedRandomSampler\', \'augmentation\': \'RandomResizedCrop(300, scale 0.80-1.00, ratio 0.90-1.10), HFlip, VFlip, RandomRotation(180), mild ColorJitter\', \'checkpoint_selection\': \'VALIDATION original-view Macro-F1; accuracy tie-breaker; earlier epoch final tie-breaker\', \'inference\': \'fixed TTA7 for all six selected checkpoints\', \'within_family_ensemble\': \'equal 1/3 probability average\', \'between_family_fusion\': \'fixed 50% B0 family + 50% B3 family\', \'threshold\': \'argmax\', \'calibration\': \'none\', \'weight_search\': \'none\'}, \'test_policy\': {\'test_used_for_training\': False, \'test_used_for_checkpoint_selection\': False, \'test_used_for_tta_selection\': False, \'test_used_for_weight_selection\': False, \'test_evaluated_once_after_system_lock\': True, \'post_test_model_change_allowed\': False}, \'historical_context\': {\'paperexp01c_clean_reference\': {\'accuracy\': 0.8682, \'macro_f1\': 0.7333, \'balanced_accuracy\': 0.7076, \'melanoma_recall\': 0.7524, \'macro_auc\': 0.9756}, \'comparison_note\': \'Clean and conventional test partitions are different resplits of the same historically used source dataset. Compare descriptively; do not claim a paired independent test.\'}}\n    (OUT / "paper_exp04a_protocol.json").write_text(\n        json.dumps(protocol_snapshot, indent=2), encoding="utf-8"\n    )\n    try:\n        shutil.copy2(Path(__file__), OUT / "paper_exp04a_code_snapshot.py")\n    except Exception:\n        pass\n\n    build_material_index()\n\n    results_zip = WORKING_ROOT / "PAPEREXP04A_Conventional_ImageLevel_Results.zip"\n    recovery_zip = WORKING_ROOT / "PAPEREXP04A_Conventional_ImageLevel_Recovery.zip"\n\n    with zipfile.ZipFile(results_zip, "w", zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(OUT.rglob("*")):\n            if p.is_file() and p.suffix.lower() not in {".pth", ".pt", ".ckpt"}:\n                z.write(p, arcname=str(p.relative_to(OUT)))\n\n    with zipfile.ZipFile(recovery_zip, "w", zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(OUT.rglob("*")):\n            if p.is_file():\n                z.write(p, arcname=str(p.relative_to(OUT)))\n\n    print("\\n" + "=" * 116)\n    print("PAPER-EXP-04A COMPLETE — CONVENTIONAL IMAGE-LEVEL INTERNAL EVALUATION")\n    print("=" * 116)\n    print("VALIDATION:", json.dumps(val_metrics, indent=2))\n    print("FROZEN TEST:", json.dumps(test_metrics, indent=2))\n    print("\\nLESION-OVERLAP SUBGROUP PERFORMANCE:")\n    print(subgroup.to_string(index=False))\n    print("\\nPOST-TEST TUNING ALLOWED:", False)\n    print("\\nUPLOAD BOTH:")\n    print(results_zip)\n    print("SHA256:", sha256_file(results_zip))\n    print(recovery_zip)\n    print("SHA256:", sha256_file(recovery_zip))\n\n\nif __name__ == "__main__":\n    main()\n'
p = Path("/kaggle/working/PAPEREXP04A_conventional_imagelevel_b0b3_300.py")
p.write_text(SCRIPT, encoding="utf-8")
print("Wrote:", p)
print("Code version:", 'PAPEREXP04A_CONVENTIONAL_IMAGE_LEVEL_B0B3_300_FIXED_FUSION_2026-08-24_v1')
print("Script bytes:", p.stat().st_size)


Wrote: /kaggle/working/PAPEREXP04A_conventional_imagelevel_b0b3_300.py
Code version: PAPEREXP04A_CONVENTIONAL_IMAGE_LEVEL_B0B3_300_FIXED_FUSION_2026-08-24_v1
Script bytes: 51827


In [2]:
import subprocess, sys

cmd = [sys.executable, "-u", "/kaggle/working/PAPEREXP04A_conventional_imagelevel_b0b3_300.py"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


Running: /usr/bin/python3 -u /kaggle/working/PAPEREXP04A_conventional_imagelevel_b0b3_300.py
[GPU] Tesla T4
[CODE VERSION] PAPEREXP04A_CONVENTIONAL_IMAGE_LEVEL_B0B3_300_FIXED_FUSION_2026-08-24_v1
[SCIENTIFIC STATUS] conventional image-level split; intentionally NON-lesion-safe
[WEIGHT] downloading official Torchvision weight: efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 172MB/s]


[WEIGHT] downloading official Torchvision weight: efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 151MB/s]
B0-300 seed 20260820 epoch 2/30:   0%|          | 0/126 [00:00<?, ?it/s]

{"family": "B0", "seed": 20260820, "epoch": 1, "train_loss": 1.2331400665956245, "lr_before_scheduler_step": 0.0001, "accuracy": 0.7694524495677233, "macro_f1": 0.6259529767918414, "balanced_accuracy": 0.6440226498885372, "melanoma_recall": 0.7586206896551724, "macro_auc": 0.9535250509896506}


...

{"family": "B3", "seed": 20260822, "epoch": 29, "train_loss": 0.2937772879150083, "lr_before_scheduler_step": 6.03798896514423e-06, "accuracy": 0.8962536023054755, "macro_f1": 0.8365924860936022, "balanced_accuracy": 0.8082168335374104, "melanoma_recall": 0.8362068965517241, "macro_auc": 0.9843374851547975}


{"family": "B3", "seed": 20260822, "epoch": 30, "train_loss": 0.2886872259092587, "lr_before_scheduler_step": 5.260209970007014e-06, "accuracy": 0.8933717579250721, "macro_f1": 0.8178549682106852, "balanced_accuracy": 0.7833791245615619, "melanoma_recall": 0.8275862068965517, "macro_auc": 0.9829191815741348}


TEST lesion bootstrap: 100%|██████████| 2000/2000 [00:23<00:00, 83.61it/s]



PAPER-EXP-04A COMPLETE — CONVENTIONAL IMAGE-LEVEL INTERNAL EVALUATION
VALIDATION: {
  "accuracy": 0.9260326609029779,
  "macro_f1": 0.8758812231144325,
  "balanced_accuracy": 0.8476517413516353,
  "melanoma_recall": 0.8275862068965517,
  "macro_auc": 0.9931287707470143
}
FROZEN TEST: {
  "accuracy": 0.913491246138002,
  "macro_f1": 0.867605362158862,
  "balanced_accuracy": 0.8646667352563578,
  "melanoma_recall": 0.8055555555555556,
  "macro_auc": 0.9876583695539498
}

LESION-OVERLAP SUBGROUP PERFORMANCE:
                               subset  images  unique_lesions  accuracy  macro_f1  balanced_accuracy  melanoma_recall  macro_auc
                             test_all     971             945  0.913491  0.867605           0.864667         0.805556   0.987658
                 lesion_seen_in_train     372             360  0.908602  0.926032           0.926485         0.885714   0.993889
             lesion_not_seen_in_train     599             585  0.916528  0.796137           0.789615 

CompletedProcess(args=['/usr/bin/python3', '-u', '/kaggle/working/PAPEREXP04A_conventional_imagelevel_b0b3_300.py'], returncode=0)

In [3]:
from pathlib import Path
import hashlib

for name in [
    "PAPEREXP04A_Conventional_ImageLevel_Results.zip",
    "PAPEREXP04A_Conventional_ImageLevel_Recovery.zip",
]:
    p = Path("/kaggle/working") / name
    print("\nUPLOAD THIS FILE:")
    print(p)
    print("Exists:", p.exists())
    if p.exists():
        print("Size:", p.stat().st_size)
        print("SHA256:", hashlib.sha256(p.read_bytes()).hexdigest())



UPLOAD THIS FILE:
/kaggle/working/PAPEREXP04A_Conventional_ImageLevel_Results.zip
Exists: True
Size: 2653190
SHA256: 8d90a6a48b700761390b4e0efb662316abba9c290d05fdd2b76584518076c472

UPLOAD THIS FILE:
/kaggle/working/PAPEREXP04A_Conventional_ImageLevel_Recovery.zip
Exists: True
Size: 168395066
SHA256: 6e34dbd798d3785a0f14029fa8a1a0b92b84dcdf124f852040e47fac5c37c5f8
